# TOPIC 795 — THE LITERARY SECOND BRAIN
## From a library of books to an auditable atlas of ideas

**A pedagogical laboratory in institutional-style literary memory, semantic navigation, and bounded AI judgment**  
**Source corpus:** `Atlas_Literario_Estructural.zip`  
**Root:** `/content/drive/MyDrive/Colab Notebooks/TOPIC_795 THE LITERARY VAULT`  
**Judgment model:** `claude-haiku-5-5` (optional; Colab Secrets: `ANTHROPIC_API_KEY`)

### The research problem

Imagine a reader who has assembled a personal library. The books have authors, dates, traditions, motifs, and questions about identity, mortality, freedom, power, and suffering. A typical search engine can locate a title or keyword. A **Second Brain** should do something harder: show how an existential question can lead through multiple works and concepts, preserve the precise reason for each connection, distinguish documented statements from speculative associations, and make it possible to compare two research strategies: one governed by prewritten rules, the other choosing its next move through bounded language-model judgment.

We will use the **existing structural literary atlas**, not invent a substitute corpus. The supplied ZIP has 231 nodes, 2,511 directed multigraph edges and 54 identified works, and includes 14 explicit circuits. These counts describe the provided atlas, **not** an independently verified catalogue of every book in the personal reading project. The source README warns that many relationships are provisional, that links from all works to broad questions often represent *pending evaluation*, and that some supposed author influences remain unverified. No experiment in this notebook can turn such a link into a historical fact by traversing it.

### The human analogy: a library, a cartographer and a skeptical reading committee

The **Librarian** restores the books and their references; the **Cataloguer** checks whether every citation resolves; the **Cartographer** compiles a map of directed, typed links; the **Research Coordinator** identifies starting shelves for five existential questions; the **Route Planner** proposes reproducible paths; the **Judgment Explorer** uses Claude to choose among a restricted list of permitted next books; the **Archivist** assembles evidence packets with their provenance; and the **Reading Committee** compares the findings and their uncertainty. These are *teaching roles*, not claims that ten autonomous agents are running.

### What students should be able to explain

By the end, a student should distinguish (i) Markdown vault versus compiled graph; (ii) graph adjacency versus a documented literary argument; (iii) forward relationships versus reverse inspection; (iv) deterministic scoring versus LLM choice; (v) a plausible reading route versus a validated answer; and (vi) what the evidence packet includes versus what its sources **do not** support. The notebook deliberately exposes the data and the decision points in tables, small maps, and readable reports rather than offering only a large, undecipherable network plot.

### Sequence

1. Prepare the laboratory and mount Google Drive.
2. Restore the original literary vault safely.
3. Audit the physical notes, wikilinks, and stored graph.
4. Compile thematic views and visualize a readable neighborhood.
5. Define existential questions and initial shelves.
6. Establish the librarian's travel rules and candidate scoring.
7. Run the deterministic literary journey.
8. Run optional **bounded Claude** navigation over the same permitted routes.
9. Prepare source-aware dossiers and compare what each journey discovered.
10. Produce a readable research report, full audit trace, and downloadable enhanced vault.

**Central question:** *When two investigators start with the same library and question, do they collect the same books—and does either journey justify an answer?*

> **Scope and integrity:** All demonstrations are pedagogical. The system does not assert that reading a book transforms a person's beliefs, establish verified authorial influence, invent quotations, or infer the user's life experience from the graph. A genuine before/after study would require a separate research design and evidence.

## Cell 01 — Prepare the research room

**Analogy.** Before any librarian opens a book, the reading room needs shelves, catalogues, notebooks and a place to preserve its findings. In computational terms, we mount Google Drive and establish separate folders for the source archive, extracted working vault and experiment outputs. We also initialize a reproducible random seed and an optional LLM flag.

**What enters:** a source ZIP that you will put in the Google Drive root. **What leaves:** a durable directory layout and control settings. **What we deliberately avoid:** requiring an API key just to learn vault construction and deterministic traversal. Setting `RUN_LLM=False` is a fully functional pedagogical mode; setting it to `True` enables live Claude calls, which may incur charges.

**Three controls matter.** `MODEL_ID` is the exact requested identifier; the notebook does not silently replace it. `MAX_STEPS` limits the length of a journey. `TOP_K` determines how many possible neighbors the LLM may choose from. A strong laboratory reveals these parameters before showing their effects, because they are institutional rules, not implementation trivia.

**Try this:** Change `MAX_STEPS` from 6 to 3 after completing the lab and see how the collected evidence changes. Is a shorter journey necessarily a worse one? That is a research question, not an assumption.

In [ ]:
%pip -q install networkx pandas matplotlib pyyaml anthropic
from google.colab import drive, userdata
from pathlib import Path
from collections import Counter, defaultdict
from IPython.display import display, Markdown
import os, json, zipfile, re, hashlib, shutil, random, math
import networkx as nx, pandas as pd, matplotlib.pyplot as plt
import yaml

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Colab Notebooks/TOPIC_795 THE LITERARY VAULT')
ROOT.mkdir(parents=True, exist_ok=True)
WORK = ROOT / 'workspace'; WORK.mkdir(exist_ok=True)
RESULTS = ROOT / 'results'; RESULTS.mkdir(exist_ok=True)
ZIP_PATH = ROOT / 'Atlas_Literario_Estructural.zip'
VAULT = WORK / 'Atlas_Literario_Estructural'
MODEL_ID = 'claude-haiku-5-5'
RUN_LLM = False  # Set True for live Claude traversal after validating credentials.
MAX_STEPS = 6
TOP_K = 8
RANDOM_SEED = 795
random.seed(RANDOM_SEED)
print('Root:', ROOT, '| Source ZIP:', ZIP_PATH)
print('RUN_LLM:', RUN_LLM, '| Model:', MODEL_ID, '| Steps:', MAX_STEPS)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Root: /content/drive/MyDrive/Colab Notebooks/TOPIC_795 THE LITERARY VAULT | Source ZIP: /content/drive/MyDrive/Colab Notebooks/TOPIC_795 THE LITERARY VAULT/Atlas_Literario_Estructural.zip
RUN_LLM: False | Model: claude-haiku-5-5 | Steps: 6


## Cell 02 — The Librarian restores the actual library

**What is a vault?** It is a directory of ordinary Markdown documents: a work may be a note, an author another note, and a philosophical question a third. Links such as `[[01 Obras/La metamorfosis]]` are navigable references for Obsidian. This makes the same material inspectable by a human reader *and* a Python parser. The supplied ZIP also includes `graph.json`, GraphML, tables, and Canvas maps. We **retain** those artifacts and treat them as different representations, not as competing sources to replace one another.

**The crucial distinction:** the Markdown notes are the durable textual library; `graph.json` is a compiled inventory of relationships supplied with that library. Their consistency must be examined, not presumed. We also avoid overwriting the supplied archive or inventing missing literary works.

**Safety and reproducibility.** The extraction procedure checks every ZIP member against the intended working directory to prevent path traversal and refuses symbolic-link entries. It computes an SHA-256 digest of the input archive. If two students start from the same ZIP checksum and parameters, differences in deterministic results should be explainable. For live language models, reproducibility is weaker, so we preserve each selection and its justification.

**Student checkpoint:** Open a Markdown file in Obsidian. Where does the story live? Where does a relationship live? Is an explicitly drawn line between two books sufficient evidence that one author influenced the other? No—the edge's relation and validation status both matter.

In [ ]:
if not ZIP_PATH.is_file():
    raise FileNotFoundError('Copy Atlas_Literario_Estructural.zip into: ' + str(ROOT))
archive_digest = hashlib.sha256(ZIP_PATH.read_bytes()).hexdigest()
with zipfile.ZipFile(ZIP_PATH) as z:
    for info in z.infolist():
        target = (WORK / info.filename).resolve()
        if not target.is_relative_to(WORK.resolve()):
            raise ValueError('Unsafe ZIP path: ' + info.filename)
        if (info.external_attr >> 16) & 0o170000 == 0o120000:
            raise ValueError('Symbolic link not allowed: ' + info.filename)
    if VAULT.exists(): shutil.rmtree(VAULT)
    z.extractall(WORK)
assert (VAULT/'graph.json').exists()
notes = sorted(p for p in VAULT.rglob('*.md') if p.name != 'README.md')
print('Archive SHA-256:', archive_digest)
print('Markdown note files:',len(notes),'| Canvas maps:',len(list(VAULT.rglob('*.canvas'))))
print('Example note:',notes[0].relative_to(VAULT))
print(notes[0].read_text(encoding='utf-8')[:750])

Archive SHA-256: f13911dcd06b9a0f2f4144c1e29b00866fa9cd7bdbbd2e6bf5c1740031544345
Markdown note files: 232 | Canvas maps: 14
Example note: 00 Inicio/Atlas.md
# Atlas literario estructural

## Entradas
- [[03 Preguntas/E01 Identidad]]
- [[03 Preguntas/E02 Sentido]]
- [[03 Preguntas/E03 Libertad]]
- [[03 Preguntas/E04 Moralidad]]
- [[03 Preguntas/E05 Sufrimiento]]
- [[03 Preguntas/E06 Muerte]]
- [[03 Preguntas/E07 Amor]]
- [[03 Preguntas/E08 Poder]]
- [[03 Preguntas/E09 Sociedad]]
- [[03 Preguntas/E10 Justicia]]
- [[03 Preguntas/E11 Conocimiento]]
- [[03 Preguntas/E12 Destino]]
- [[03 Preguntas/E13 Trascendencia]]
- [[03 Preguntas/E14 Transformación]]
- [[03 Preguntas/E15 Condición humana]]

## Circuitos
- [[10 Circuitos/C001 · identidad y extrañamiento]]
- [[10 Circuitos/C002 · transformación y mortalidad]]
- [[10 Circuitos/C003 · mecanismos del poder]]
- [[10 Circuitos/C004 · censura y control]]


### Clarification on Vault Nodes and Textual Richness



In this current structural literary atlas, every node in the graph maps directly to a physical Markdown document. However, many of these files function primarily as structural placeholder cards (e.g., conceptual indices, category roots, or unvalidated connection nodes) containing only YAML metadata or standard wiki-link pointers, with little to no analytical narrative body (such as the node for Love or some broad existential questions).

This highlights a crucial pedagogical theme: **the difference between a structural network template and a mature literary Second Brain**.

* **The Current State:** The vault provides a reliable structural skeleton (nodes, edges, directionality, and metadata validation flags) to prevent empty references.
* **The Opportunity for Enrichment:** A more mature vault would indeed enrich these thin or empty files. For example, opening the "Love" note should ideally display critical synthesis, curated text excerpts, historical tradition summaries, and precise academic arguments justifying its outward links. Without this narrative tissue, traversing the graph yields paths but not necessarily deep literary understanding.

## Cell 03 — The Cataloguer verifies the books and their connections

**Why can a gorgeous graph be misleading?** In our earlier central-bank laboratory, a computational graph contained edges even when the Obsidian interface displayed isolated notes. This time, we explicitly examine the *physical Markdown* and the *stored graph*. We do not simply count Python edges and declare victory.

Every note may contain YAML frontmatter and body wikilinks. The parser extracts its stable `node_id`, its `node_type`, and the targets of `[[...]]` references. It then checks whether a link target resolves to a physical file and whether the stored graph nodes match the note inventory. **However, parallel typed edges in the graph need not correspond one-for-one with visible wikilinks**: several directed relations may connect the same pair of notes, and links may be repeated. The audit therefore reports matching, missing, and duplicate representations separately.

The graph includes **uncertainty statuses**. A link tagged `hipotesis_pendiente`, `sin_analizar`, or `original_sin_validar` is not a literary fact. It is still useful as a candidate route for investigation, but must be flagged rather than presented as verified interpretive knowledge. A student should be able to answer both: *Where does this link lead?* and *Why am I allowed to believe it?*

**What enters:** actual `.md` files plus the supplied graph JSON. **What leaves:** a physical-note catalogue, a directed multigraph, and an integrity report. **Critical question:** What kinds of error are invisible if we validate only a graph compiled in memory?

### Definition and Pertinence of Wikilinks

#### 1. What is a Wikilink?
A **wikilink** is a double-bracketed syntax (e.g., `[[01 Obras/La metamorfosis]]` or `[[Target Note Name|Custom Display Text]]`) used widely in personal knowledge management wikis and note-taking systems like Obsidian.

Unlike standard HTML or markdown links, a wikilink is typically:
* **Relative and Internal:** It points to another document within the local vault directory based on its filename or path, without needing a rigid absolute filepath or web URL.
* **Bi-directional:** It enables tools to easily resolve backlink tracking, showing not only where a note points, but which other notes point back to it.

#### 2. Pertinence in this Notebook & Vault
In this literary network study, wikilinks serve as the physical, ground-truth connective tissue:
* **Bridging Text and Graph:** While `graph.json` represents a compiled, pre-calculated metadata snapshot of connections, the raw wikilinks embedded inside the `.md` files represent the actual physical connections written by the human authors or researchers.
* **Source of Audit Verification:** They allow us to audit the graph. By parsing every `[[...]]` link in our 232 Markdown files, we can verify if a link actually resolves to an existing file, discover dangling references, and check whether the stored structural edges in the graph correspond to real, readable links in the texts.

### The Connection Between Wikilinks and Wikipedia

There is a direct conceptual and historical connection between **wikilinks** and **Wikipedia**:

1. **Historical Origin:**
   The concept of wiki-style linking originates from the first wiki, Ward Cunningham's *WikiWikiWeb* (which used CamelCase text like `WikiWeb` to auto-link pages). However, **Wikipedia** standardized the double-bracketed `[[Link]]` syntax we call "wikilinks" today to make linking between articles incredibly simple and readable without needing complex HTML.

2. **The Philosophy of Easy Cross-Referencing:**
   * In **Wikipedia**, a contributor doesn't need to know the actual URL of the page "Franz Kafka". They just type `[[Franz Kafka]]` in the editor, and the mediawiki platform automatically resolves that to the target page.
   * Similarly, in personal knowledge systems like our **Obsidian Vault**, using `[[01 Obras/La metamorfosis]]` follows this exact same principle. It allows a researcher to build a private, localized mini-Wikipedia of literary analysis by simply typing the target note name inside double brackets.

### Is a "Second Brain" a Personal Wikipedia?

Yes, absolutely! That is one of the most accurate ways to conceptualize a **Second Brain** or a **digital garden**.

Here is how they align conceptually:

* **Your Private Encyclopedia:** Just as Wikipedia aims to catalog the sum of human knowledge, your Second Brain catalogs the sum of *your* knowledge, readings, thoughts, and research.
* **Decentralized, Networked Structure:** Traditional note-taking is hierarchical (folders inside folders, like a filing cabinet). A personal Wikipedia/Second Brain uses **wikilinks** to create a web of interconnected notes. If you are writing about Franz Kafka, you can link to `[[03 Preguntas/E14 Transformación]]` and `[[La metamorfosis]]` without having to decide which single "folder" the thought belongs to.
* **Emergent Structure:** Instead of planning a rigid structure beforehand, you write notes freely and link them as you go. Over time, a complex network (a graph) *emerges* naturally from these connections—just like Wikipedia grew from a few articles into millions of interconnected pages.

In [ ]:
iraw = json.loads((VAULT/'graph.json').read_text(encoding='utf-8'))
MG = nx.MultiDiGraph()
for obj in raw['nodes']:
    MG.add_node(obj['id'],**{k:v for k,v in obj.items() if k!='id'})
for obj in raw['edges']:
    if obj['source'] in MG and obj['target'] in MG:
        MG.add_edge(obj['source'],obj['target'],**{k:v for k,v in obj.items() if k not in ('source','target','key')})

def parse_note(p):
    t=p.read_text(encoding='utf-8-sig')
    match=re.match(r'^---\s*\n(.*?)\n---\s*\n',t,re.S)
    meta=(yaml.safe_load(match.group(1)) or {}) if match else {}
    links=re.findall(r'\[\[([^\]]+)\]\]',t)
    links=[x.split('|',1)[0].split('#',1)[0].strip() for x in links]
    return {'id':str(meta.get('node_id',p.relative_to(VAULT).with_suffix('').as_posix())),
            'type':str(meta.get('node_type','unknown')),'path':str(p.relative_to(VAULT)),
            'body':t,'links':links,'metadata':meta}
CATALOG={entry['id']:entry for entry in map(parse_note,notes)}
actual_links=[(n,t) for n,d in CATALOG.items() for t in d['links']]
missing_targets=sorted(set(t for _,t in actual_links if t not in CATALOG))
missing_graph_nodes=sorted(set(MG.nodes)-set(CATALOG))
missing_physical_nodes=sorted(set(CATALOG)-set(MG.nodes))
master_pairs={(u,v) for u,v in MG.edges()}
physical_pairs={(u,v) for u,v in actual_links if v in CATALOG}
AUDIT={'archive_sha256':archive_digest,'notes':len(CATALOG),'graph_nodes':len(MG),
       'graph_multiedges':MG.number_of_edges(),'wikilinks':len(actual_links),
       'unique_physical_pairs':len(physical_pairs),'unique_graph_pairs':len(master_pairs),
       'unresolved_wikilinks':missing_targets[:30],
       'graph_nodes_without_notes':missing_graph_nodes[:30],
       'notes_without_graph_nodes':missing_physical_nodes[:30],
       'graph_pairs_without_wikilink':len(master_pairs-physical_pairs),
       'wikilink_pairs_without_graph_edge':len(physical_pairs-master_pairs)}
print(json.dumps(AUDIT,indent=2,ensure_ascii=False))
print('Relationship status distribution:',Counter(d.get('status','unspecified') for _,_,d in MG.edges(data=True)).most_common(9))
(RESULTS/'source_integrity_audit.json').write_text(json.dumps(AUDIT,indent=2,ensure_ascii=False),encoding='utf-8')
assert len(MG)>0 and len(CATALOG)>0

{
  "archive_sha256": "f13911dcd06b9a0f2f4144c1e29b00866fa9cd7bdbbd2e6bf5c1740031544345",
  "notes": 232,
  "graph_nodes": 231,
  "graph_multiedges": 2511,
  "wikilinks": 2564,
  "unique_physical_pairs": 1836,
  "unique_graph_pairs": 1807,
  "unresolved_wikilinks": [],
  "graph_nodes_without_notes": [],
  "notes_without_graph_nodes": [
    "00 Inicio/Atlas"
  ],
  "graph_pairs_without_wikilink": 0,
  "wikilink_pairs_without_graph_edge": 29
}
Relationship status distribution: [('sin_analizar', 810), ('original_sin_validar', 588), ('metadato', 213), ('hipotesis_pendiente', 210), ('hipotesis_interpretativa', 134), ('no_evaluada', 120), ('ruta_de_navegacion_propuesta', 112), ('hipotesis_historica', 100), ('estructural', 56)]


## Cell 04 — The Cartographer draws usable maps, not a cloud of dots

A 231-node graph may be visually impressive and pedagogically useless. Imagine spreading the whole library's cross-references across one wall: every book is present, but students cannot see a meaningful reading journey. Our solution is to distinguish the **master multigraph**, which preserves parallel typed relationships, from a simpler **navigation graph**, and from a small, **question-centered neighborhood**.

**The road-map analogy.** A work-to-author edge is a bibliographic road; a concept-to-question edge is a thematic road; a circuit edge is a proposed reading itinerary. A single book may appear on several overlapping maps. The master graph preserves all recorded routes, while a thematic projection limits the roads displayed. This is not evidence that all road types have equal reliability. A reverse-traversed edge may be useful for exploration without reversing the historical claim in the source.

We draw only a small neighborhood around a chosen book or question and report its actual relationship labels. The function accepts a node ID, hop limit, and maximum node count. It returns the exact subgraph plotted, so the student can inspect nodes and edges rather than trusting the image. A second table describes the relation types in the selected neighborhood.

**Crucial evaluation:** Connectivity establishes a potential research pathway—not an interpretation, and certainly not proof of literary influence. The number of connected components tells us something about structure, not about the truth of the hypotheses inside that structure.

**Question for the next cell:** With many overlapping maps available, where should our investigation begin if the problem concerns identity or mortality?

In [ ]:
G=nx.DiGraph();G.add_nodes_from(MG.nodes(data=True))
for u,v,d in MG.edges(data=True):
    if not G.has_edge(u,v):G.add_edge(u,v,relations=set(),statuses=set())
    G[u][v]['relations'].add(str(d.get('relation','unspecified')))
    G[u][v]['statuses'].add(str(d.get('status','unspecified')))
for u,v in G.edges():
    G[u][v]['relations']=sorted(G[u][v]['relations'])
    G[u][v]['statuses']=sorted(G[u][v]['statuses'])
NAV=G.to_undirected()
def label(n):return str(G.nodes[n].get('label',n.split('/')[-1]))
def kind(n):
    k=str(G.nodes[n].get('kind','')).casefold()
    if not k:k=n.split('/')[0].casefold()
    return k

def show_neighborhood(center, hops=1, max_nodes=22, show_edge_types=True):
    if center not in G:raise ValueError('Unknown node: '+center)
    distances=nx.single_source_shortest_path_length(NAV,center,cutoff=int(hops))
    ordered=sorted(distances, key=lambda x:(distances[x],-NAV.degree(x),x))[:int(max_nodes)]
    H=G.subgraph(ordered).copy(); pos=nx.spring_layout(H,seed=795,iterations=160)
    types=sorted({kind(n) for n in H});colors={k:plt.cm.tab20(i%20) for i,k in enumerate(types)}
    fig,ax=plt.subplots(figsize=(13,8))
    nx.draw_networkx_edges(H,pos,alpha=.32,arrows=True,arrowsize=10,ax=ax)
    for t in types:
        nodes_t=[v for v in H if kind(v)==t]
        nx.draw_networkx_nodes(H,pos,nodelist=nodes_t,node_color=[colors[t]],node_size=350 if len(H)<15 else 230,label=t,ax=ax)
    nx.draw_networkx_labels(H,pos,labels={v:label(v)[:20] for v in H},font_size=8,ax=ax)
    if show_edge_types and H.number_of_edges()<=14:
        nx.draw_networkx_edge_labels(H,pos,edge_labels={(u,v):'/'.join(d['relations'][:1]) for u,v,d in H.edges(data=True)},font_size=6,rotate=False,ax=ax)
    ax.set_title('Literary atlas: '+label(center)+' | '+str(len(H))+' notes, '+str(H.number_of_edges())+' directed pairs')
    ax.legend(loc='upper left',fontsize=8,ncol=2);ax.axis('off');plt.tight_layout();plt.show()
    return H
WORK_NODES=sorted([x for x in G if x.startswith('01 Obras/')])
print('Graph:',len(G),'nodes,',G.number_of_edges(),'unique directed pairs,',MG.number_of_edges(),'original edges')
print('Works:',len(WORK_NODES),'| Weak components:',nx.number_weakly_connected_components(G))
example=next((x for x in WORK_NODES if 'metamorfosis' in x.casefold()),WORK_NODES[0])
H_EXAMPLE=show_neighborhood(example,hops=1,max_nodes=18)
display(pd.DataFrame([{'source':label(u),'relations':', '.join(d['relations']),'target':label(v),'status':', '.join(d['statuses'])} for u,v,d in H_EXAMPLE.edges(data=True)]).head(20))

In [ ]:
import pandas as pd
from collections import Counter

# Helper function defined inline to guarantee execution safety
def get_node_kind(n):
    k = str(MG.nodes[n].get('kind', '')).casefold() if n in MG.nodes else ''
    if not k:
        k = n.split('/')[0].casefold()
    return k

# Analyze the most common flows by looking at source node kind -> relation -> target node kind
flows = []
for u, v, d in MG.edges(data=True):
    source_kind = get_node_kind(u)
    target_kind = get_node_kind(v)
    relation = d.get('relation', 'unspecified')
    flows.append((source_kind, relation, target_kind))

flow_counts = Counter(flows).most_common(15)

# Create a structured DataFrame for the common flows
flow_table = pd.DataFrame([
    {
        'Source Type': f[0],
        'Relationship': f[1],
        'Target Type': f[2],
        'Frequency': count
    } for f, count in flow_counts
])

print("Top 15 Most Common Edge Flows in the Master Graph:")
display(flow_table)

Top 15 Most Common Edge Flows in the Master Graph:


,Source Type,Relationship,Target Type,Frequency
0,obra,EVALUACION_PENDIENTE,03 preguntas,600
1,03 preguntas,enlace_original,01 obras,210
2,01 obras,enlace_original,03 preguntas,210
3,01 obras,PREGUNTA_CANDIDATA,03 preguntas,210
4,01 obras,EVALUACION_PENDIENTE,03 preguntas,210
5,obra,PREGUNTA_CANDIDATA,03 preguntas,120
6,autor,ASOCIADO_A_MOVIMIENTO,movimiento,45
7,03 preguntas,RELACION_CONCEPTUAL,03 preguntas,40
8,obra,ASOCIADA_A_MOVIMIENTO,movimiento,40
9,obra,PERTENECE_A_GENERO,genero,39


### Node Types (Kinds) in the Master Graph

Based on the architecture of our literary atlas, each node is categorized into a specific "kind" (type) which defines its role and meaning in the network:

| Node Type (`kind`) | Meaning & Role in the Atlas | Examples in the Vault |
| :--- | :--- | :--- |
| **`obra`** (Work) | Represents a specific literary work (book, essay, play, or short story) being analyzed. | *La metamorfosis*, *El proceso* |
| **`autor`** (Author) | Represents the writer of the literary works, housing biographical context and overarching stylistic links. | *Franz Kafka* |
| **`concepto`** (Concept) | Represents a thematic abstraction, philosophical motif, or key literary device observed across texts. | *Aliensación*, *Culpa*, *Absurdo* |
| **`pregunta`** (Question) | Represents one of the core investigative, existential inquiries driving our research itinerary. | *Who am I? (Identity)*, *What is suffering?* |
| **`movimiento`** (Movement) | Represents a historical, philosophical, or artistic movement that contextualizes authors and works. | *Existencialismo*, *Expresionismo* |
| **`genero`** (Genre) | Denotes the traditional classification of literary forms. | *Novela*, *Ensayo* |
| **`circuito`** (Circuit) | Represents a curated reading itinerary or a proposed path of traversal through the graph designed by instructors. | *Ruta existencialista* |
| **`relacion`** (Relation) | Meta-nodes or nodes representing specific relational structures or connections. | *Relación conceptual* |

## Cell 05 — The Research Coordinator chooses the starting library cards

**The question comes before the search.** Rather than arbitrarily entering the network at a famous author, we begin with five existential questions already represented in the source atlas: identity, suffering, freedom, power, and mortality. The questions are *investigative prompts*, not claims that every book has been shown to answer them.

The Research Coordinator uses concept labels to locate initial records. In the library analogy, a seed is a card identifying a shelf from which an investigative journey can begin. The card is not itself a quotation, and a match on the word *identity* does not establish that an author's treatment of identity fits the question. It is merely an entrance into the catalogue.

We intentionally display every question, its matched seed nodes, and any unmatched terms. This allows the student to interrogate the selection process before trusting downstream results. The initial seed ranking privileges question records over generic conceptual matches, but students should ask whether a question node can be useful if the graph simply connects it to every work through a pending-evaluation edge.

**Two experimental pathways.** Both methods will receive the same starting seed and the same atlas. Route A uses explicit scoring rules; Route B asks Claude to choose the next permissible neighbor. Neither method receives permission to invent works or leave the actual graph. Their paths are comparable precisely because the difference in decision policy is visible.

**Question for the next cell:** Once the starting shelf is fixed, what rules should determine which reference we follow next—and how should uncertain relationships affect our choices?

In [ ]:
QUESTIONS=[
 ('Q1','Who am I, and what constitutes my identity?',['identidad']),
 ('Q2','What meaning does human suffering have?',['sufrimiento']),
 ('Q3','Are we truly free to make decisions?',['libertad','libre albedrío']),
 ('Q4','How does power transform people?',['poder']),
 ('Q5','How should we confront our mortality?',['muerte','mortalidad'])]
def find_seeds(terms):
    found=[]
    for node in G:
        if any(t in label(node).casefold() for t in terms):
            priority=0 if 'pregunta' in kind(node) else 1
            found.append((priority,len(label(node)),node))
    return [x[-1] for x in sorted(found)[:4]]
SEEDS={qid:find_seeds(terms) for qid,_,terms in QUESTIONS}
seed_table=pd.DataFrame([{'ID':qid,'Question':q,'Starting notes':', '.join(label(x) for x in SEEDS[qid]),
                          'Count':len(SEEDS[qid])} for qid,q,_ in QUESTIONS])
display(seed_table)
assert all(SEEDS.values()),'At least one question has no seed: inspect the source labels.'

## Cell 06 — The Route Planner defines transparent travel rules

Think of the five existential questions as five destinations and the initial seeds as our starting cities. We need a rule for moving through the library's overlapping roadmaps. A simple graph search would follow links indiscriminately. A better *teaching* experiment explains exactly why one neighboring book or concept is preferred over another.

The deterministic ranking policy assigns positive points for valuable node categories (works, authors, questions, concepts), a small reward for encountering an unvisited category, and a match bonus when the neighboring label shares the question's key terms. It penalizes edges whose only apparent support is `EVALUACION_PENDIENTE`, `PREGUNTA_CANDIDATA`, or `enlace_original`. Those weights are **instructor-designed**, not empirical estimates of literary quality. The point is to expose a decision rule that students can critique.

**Directionality matters.** A search may inspect inbound and outbound references to discover a useful neighbor, but it records if a directed relation was followed in reverse. Reverse inspection is a navigation operation, not the assertion of a reversed causal or historical claim. In a literary atlas, confusing those two would be particularly misleading for authorial influence.

A travel log for each step includes the candidates considered, their scores, the chosen node, whether the traversal went forward or backward, and the typed statuses available for that connection. The stopping condition is bounded by `MAX_STEPS`; therefore the model cannot wander indefinitely.

**Challenge:** Replace the penalty for unverified associations with zero. Does the route become more relevant or simply more connected? This is a good example of how graph density and evidence quality can diverge.

**Question for the next cell:** Can a disciplined, entirely deterministic explorer collect meaningful books without a language model—and what will it miss?

In [ ]:
CATEGORY_WEIGHTS={'obra':3.0,'autor':2.3,'concepto':1.9,'pregunta':1.7,'relacion':1.3,'movimiento':1.0,'genero':.8,'circuito':1.1}
UNVERIFIED={'EVALUACION_PENDIENTE','PREGUNTA_CANDIDATA','enlace_original'}
def simple_kind(node):
    s=kind(node)
    for key in CATEGORY_WEIGHTS:
        if key in s:return key
    return 'other'
def connection(a,b):
    f=G.get_edge_data(a,b) or {};r=G.get_edge_data(b,a) or {}
    return {'direction':'forward' if f else 'reverse',
            'forward_relations':f.get('relations',[]),'reverse_relations':r.get('relations',[]),
            'forward_statuses':f.get('statuses',[]),'reverse_statuses':r.get('statuses',[])}
def rank_neighbors(current,visited,terms,limit=TOP_K):
    result=[]
    for nxt in NAV.neighbors(current):
        if nxt in visited:continue
        c=connection(current,nxt)
        rels=set(c['forward_relations']+c['reverse_relations'])
        pending=bool(rels) and all(r in UNVERIFIED for r in rels)
        bonus=2 if any(t in label(nxt).casefold() for t in terms) else 0
        novelty=1.1 if simple_kind(nxt) not in {simple_kind(v) for v in visited} else 0
        score=CATEGORY_WEIGHTS.get(simple_kind(nxt),.6)+bonus+novelty-(2.3 if pending else 0)
        result.append({'id':nxt,'label':label(nxt),'score':round(score,3),'connection':c,'provisional':pending})
    return sorted(result,key=lambda d:(-d['score'],d['id']))[:limit]
probe=SEEDS['Q1'][0]
print('Starting shelf:',label(probe))
display(pd.DataFrame([{'candidate':d['label'],'score':d['score'],'direction':d['connection']['direction'],
                       'provisional_only':d['provisional']} for d in rank_neighbors(probe,{probe},QUESTIONS[0][2])]))

## Cell 07 — Agent A explores the shelves deterministically

**The first journey.** Our parametric researcher starts at the first seed, inspects all eligible neighboring nodes, scores them with the published function, selects the highest-scoring unvisited neighbor, and repeats. Every candidate list is retained. In this sense the algorithm does not *understand* a novel; it operates on the structural information and metadata exposed by the atlas.

The path is not the same as a list of recommendations. It may contain a question, a genre, a concept, an author, and a work. Nor does a short route prove that two books offer substantively comparable arguments. In a dense graph—particularly one with pending links to all 15 existential questions—a shallow route may look persuasive while containing very little verified interpretive substance.

**The analyst's playbook:** receive the initial library card; open the note; enumerate outgoing and incoming references; discard already visited records; score the remaining alternatives; select one; log the decision; stop after the allotted steps. The product is a transparent itinerary, not yet an answer to the existential question.

We display a journey for each question and create a detailed table for the first. The student can inspect the score, relation direction and provisional status at every decision. There is no hidden model judgment in this stage.

**Question for the next cell:** Suppose a second investigator follows the same authorized shelves but is allowed to weigh their intellectual meaning. Would it take the same route? How should we control what it is permitted to see?

In [ ]:
def deterministic_journey(qid,question,terms):
    seeds=SEEDS[qid]
    if not seeds:return {'qid':qid,'question':question,'engine':'deterministic','path':[],'trace':[]}
    path=[seeds[0]];trace=[]
    for step in range(MAX_STEPS):
        opts=rank_neighbors(path[-1],set(path),terms)
        if not opts:break
        chosen=opts[0]
        trace.append({'step':step+1,'from':path[-1],'to':chosen['id'],'score':chosen['score'],
                      'relation':chosen['connection'],'alternatives':opts,'reason':'Top allowed deterministic score'})
        path.append(chosen['id'])
    return {'qid':qid,'question':question,'engine':'deterministic','path':path,'trace':trace}
DETERMINISTIC=[deterministic_journey(qid,q,terms) for qid,q,terms in QUESTIONS]
for trip in DETERMINISTIC:
    print('\n'+trip['qid']+' — '+trip['question'])
    print(' → '.join(label(n) for n in trip['path']))
display(pd.DataFrame([{'step':x['step'],'from':label(x['from']),'to':label(x['to']),
                       'score':x['score'],'direction':x['relation']['direction'],
                       'provisional':next((a['provisional'] for a in x['alternatives'] if a['id']==x['to']),False)}
                      for x in DETERMINISTIC[0]['trace']]))

## Cell 08 — Agent B uses bounded Claude judgment to choose the next book

Here is the key experimental comparison. Agent A followed a fixed scoring formula. Agent B sees the same question and current path, but may use Claude Haiku 5.5 to select *one of the allowed neighboring IDs*. The model receives candidate titles, kinds, relation directions, and uncertainty statuses. It does **not** receive carte blanche to invent a new book, invent an edge, or treat a provisional relationship as a verified fact.

**Agentic responsibility is divided.** The LLM *proposes* its next reading stop and explains why. Deterministic code *authorizes* that choice by checking membership in the supplied candidate list. If the model's response is invalid, the code records the error and takes the deterministic fallback. The audit distinguishes valid LLM choices from substitutions. This is not autonomous browsing of the internet; it is **bounded navigation through a closed, supplied graph**.

The system uses `ANTHROPIC_API_KEY` from Colab Secrets. `RUN_LLM=False` performs a no-API demonstration. Turning it on may result in API charges. The model identifier is shown explicitly, `claude-haiku-5-5`; if unavailable in your account, the notebook stops and asks you to change the setting rather than silently substituting another model. Calls omit sampling-temperature parameters, making the interface more robust across model variants.

A **critical epistemic limit:** selecting a better-looking graph path is not equivalent to understanding the full literary texts. The source corpus contains metadata and notes, not necessarily authenticated quotations or complete works. We instruct Claude to distinguish metadata-supported claims from its interpretations and to say when the corpus is insufficient.

**Question for the next cell:** Once two agents return with different reading journeys, how do we compare their collections without pretending that the longer or more diverse journey is necessarily more truthful?

In [ ]:
def parse_model_json(t):
    t=re.sub(r'^```(?:json)?\s*|\s*```$','',t.strip())
    try:return json.loads(t)
    except json.JSONDecodeError:
        m=re.search(r'\{.*\}',t,re.S)
        if not m:raise
        return json.loads(m.group(0))

def llm_journey(qid,question,terms,client):
    path=[SEEDS[qid][0]];trace=[]
    for step in range(MAX_STEPS):
        opts=rank_neighbors(path[-1],set(path),terms)
        if not opts:break
        allowed={x['id'] for x in opts}
        payload={'question':question,'current_node':path[-1],'path_so_far':path,
                 'allowed_candidates':[{'id':x['id'],'label':x['label'],'kind':simple_kind(x['id']),
                                        'score':x['score'],'relation':x['connection'],'provisional':x['provisional']} for x in opts]}
        system=('You are a cautious literary research navigator. Choose exactly ONE next_id from allowed_candidates. '
                'Do not invent authors, works, quotations or historical influence. Provisional links are hypotheses. '
                'Return one JSON object: {"next_id":"...","reason":"...","confidence":0.0}.')
        response=client.messages.create(model=MODEL_ID,max_tokens=650,system=system,
              messages=[{'role':'user','content':json.dumps(payload,ensure_ascii=False)}])
        raw_text='\n'.join(b.text for b in response.content if getattr(b,'type',None)=='text')
        try:
            choice=parse_model_json(raw_text);nxt=choice['next_id']
            if nxt not in allowed:raise ValueError('Requested ID not in authorized neighbors')
            valid=True;reason=str(choice.get('reason',''));confidence=choice.get('confidence')
        except (ValueError,KeyError,TypeError,json.JSONDecodeError) as exc:
            nxt=opts[0]['id'];valid=False;reason='Fallback after invalid response: '+str(exc);confidence=None
        trace.append({'step':step+1,'from':path[-1],'to':nxt,'score':next(x['score'] for x in opts if x['id']==nxt),
                      'relation':connection(path[-1],nxt),'alternatives':opts,'reason':reason,
                      'valid_model_choice':valid,'confidence':confidence})
        path.append(nxt)
    return {'qid':qid,'question':question,'engine':'bounded_claude','path':path,'trace':trace}

LLM_JOURNEYS=[];LLM_STATUS='Disabled — deterministic demonstration only'
if RUN_LLM:
    try:
        from anthropic import Anthropic
        key=userdata.get('ANTHROPIC_API_KEY') or os.environ.get('ANTHROPIC_API_KEY')
        if not key:raise RuntimeError('Set ANTHROPIC_API_KEY in Colab Secrets first.')
        client=Anthropic(api_key=key,timeout=90,max_retries=2)
        # Do not silently substitute model IDs.
        LLM_JOURNEYS=[llm_journey(qid,q,terms,client) for qid,q,terms in QUESTIONS]
        LLM_STATUS='Claude inference executed: '+MODEL_ID
    except Exception as exc:
        LLM_STATUS='LLM unavailable: '+repr(exc)
        print(LLM_STATUS,'\nSet RUN_LLM=False for deterministic mode or correct the model/key and rerun.')
else:
    print('LLM navigation is disabled. Set RUN_LLM=True in Cell 01 and rerun Cell 08.')
print(LLM_STATUS)
for trip in LLM_JOURNEYS:
    print(trip['qid'],' → '.join(label(n) for n in trip['path']))

## Cell 09 — The Archivist builds evidence packets; the Reading Committee compares journeys

Our investigators now return from the library. They may have collected different books and concepts. **Do we reward the journey that visited the most works?** Not necessarily. The real task is to evaluate the *provenance* of the collected records, the types and directions of the connecting references, and the status of each claim. We preserve each complete path plus an explicit summary of every step's relation and uncertainty.

**The Christmas-tree-light analogy.** Retrieved books are not loose sheets: the traversal carries the wiring of their connections. Yet connectivity does not prove interpretive significance. An edge whose only status is `sin_analizar` should remain explicitly tentative in the packet. We therefore compute a transparent *pending-link share*, count works, report node-category diversity and compare the two routes using Jaccard overlap. These are descriptions of the journey, **not quality scores for the literature**.

The Reading Committee may also ask Claude for a provisional synthesis, but that would be a **new analytical decision with its own evidence boundary**. Here, for clarity, the stage focuses on assembling and comparing evidence rather than claiming that a path alone has yielded a meaningful existential answer.

Each packet includes node IDs, short labels, types, and path steps with direction and edge statuses. If a note contains a substantial body, we include a *short excerpt* identified as a vault excerpt, not as a quotation from the underlying published novel. The original vault sources may be incomplete and some links have not been independently verified.

**Question for the final cell:** What can we responsibly say about the difference between the two journeys, and how do we preserve enough records for a skeptical student to reproduce the experiment?

In [ ]:
def evidence_packet(trip):
    p=trip.get('path',[])
    docs=[]
    for node in p:
        text=CATALOG.get(node,{}).get('body','')
        # Note text is vault content; not necessarily a primary literary quotation.
        excerpt=re.sub(r'(?s)^---.*?---\s*','',text).strip()[:400]
        docs.append({'id':node,'label':label(node),'kind':simple_kind(node),
                     'vault_excerpt':excerpt,'epistemic_warning':'Vault note; not verified primary-text quotation'})
    steps=[]
    for item in trip.get('trace',[]):
        e=item['relation'];rels=e['forward_relations']+e['reverse_relations']
        statuses=e['forward_statuses']+e['reverse_statuses']
        steps.append({'step':item['step'],'source':item['from'],'target':item['to'],
                      'direction':e['direction'],'relations':rels,'statuses':statuses,
                      'reason':item.get('reason',''),'model_choice_valid':item.get('valid_model_choice')})
    return {'question':trip.get('question'),'engine':trip['engine'],'sources':docs,'paths':steps,
            'limitations':['Graph links are not primary-text evidence','Pending links need human validation',
                           'No causal or literary interpretation follows from connectivity alone']}
ALL_JOURNEYS=DETERMINISTIC+LLM_JOURNEYS
for trip in ALL_JOURNEYS:trip['packet']=evidence_packet(trip)
def metric(trip):
    steps=trip['packet']['paths'];pending=sum(bool(s['statuses']) and all(x in
                {'sin_analizar','hipotesis_pendiente','original_sin_validar','metadato_demo'} for x in s['statuses']) for s in steps)
    return {'question':trip['qid'],'engine':trip['engine'],'path_length':len(trip['path']),
            'works':sum(simple_kind(x)=='obra' for x in trip['path']),
            'types':len({simple_kind(x) for x in trip['path']}),
            'provisional_step_share':round(pending/max(len(steps),1),3),
            'valid_llm_steps':sum(s.get('model_choice_valid') is True for s in steps)}
METRICS=pd.DataFrame([metric(x) for x in ALL_JOURNEYS]);display(METRICS)
for qid,_,_ in QUESTIONS:
    matched=[x for x in ALL_JOURNEYS if x['qid']==qid]
    if len(matched)==2:
        a,b=[set(x['path']) for x in matched]
        print(qid,'Jaccard route overlap:',round(len(a&b)/max(1,len(a|b)),3))
print('Packets prepared:',len(ALL_JOURNEYS),'| Source provenance preserved in packet["paths"]')

## Cell 10 — Publish an auditable reading dossier, not a fabricated interpretation

A teaching laboratory should conclude with something students can **read, challenge, and reproduce**. We therefore produce a human-readable Markdown report for each research question, a table comparing methods, machine-readable traces, a manifest of parameters and source digests, and an enhanced Obsidian ZIP. The report contains the paths traveled and selected excerpts from the vault, but does **not** pretend that a graph establishes what the full literary works say.

**The output question:** Suppose two students use the same atlas but different route policies. What is similar between their discoveries? Which steps are supported by explicit, validated information, and which are merely hypotheses? What additional reading, textual evidence, or scholarly citation would be required before drawing a persuasive conclusion about identity, suffering, freedom, power or mortality?

We also make the source limitations impossible to overlook: the provided atlas identifies 54 works, not a verified complete 84-work inventory; some connections are pending evaluation; no pre-/post-reading psychological effect has been measured; and model-generated explanations can still be inaccurate despite the allowlist. These qualifications strengthen the educational experiment, because they teach students to distinguish epistemic responsibility from software execution.

**Next research frontier.** Our initial journey can be extended into a governed adaptive research loop: the LLM identifies a missing piece of evidence, proposes which authorized source to inspect next, receives approval, and revises its provisional interpretation with a new audit trail. That would be a genuine *new* experiment, not a capability we silently attribute to this notebook.

In [ ]:
LAB=VAULT/'15 Traversal Laboratory';LAB.mkdir(exist_ok=True)
def wiki(node):return '[['+node+']]'
index=['# Literary Traversal Laboratory','',
       '> Experimental evidence routes. A graph connection is not proof of historical influence or a validated literary argument.','']
for trip in ALL_JOURNEYS:
    filename=f"{trip['qid']}_{trip['engine']}.md"
    rows=[f"# {trip['qid']} — {trip['engine']}",'',f"## Research question\n{trip['question']}",'',
          '## Connected library itinerary',' → '.join(wiki(x) for x in trip['path']),'',
          '## Auditable itinerary']
    for s in trip['packet']['paths']:
        rows += [f"### Hop {s['step']}",f"- {wiki(s['source'])} → {wiki(s['target'])}",
                 f"- Inspection direction: `{s['direction']}`",
                 f"- Typed relations: `{', '.join(s['relations'])}`",
                 f"- Evidence statuses: `{', '.join(s['statuses'])}`",
                 f"- Selection reason: {s['reason']}",'']
    rows+=['## Vault excerpts (not authenticated quotations)','']
    for source in trip['packet']['sources']:
        rows += [f"### {wiki(source['id'])}",source['vault_excerpt'][:280],'']
    rows+=['## Research judgment','The path identifies records to read and verify. The supplied graph alone does not establish a defensible answer to the existential question.','']
    (LAB/filename).write_text('\n'.join(rows),encoding='utf-8')
    index.append(f'- [[15 Traversal Laboratory/{filename[:-3]}]]')
(LAB/'Indice.md').write_text('\n'.join(index),encoding='utf-8')
METRICS.to_csv(RESULTS/'comparative_metrics.csv',index=False)
(RESULTS/'journeys_and_packets.json').write_text(json.dumps(ALL_JOURNEYS,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
MANIFEST={'archive_sha256':archive_digest,'vault_notes':len(CATALOG),'graph_nodes':len(MG),
          'graph_multiedges':MG.number_of_edges(),'model':MODEL_ID if RUN_LLM else None,
          'llm_status':LLM_STATUS,'max_steps':MAX_STEPS,'top_k':TOP_K,
          'source_limits':['54 identified works in supplied atlas, not the full 84-work list',
                           'unverified and pending literary relations retained as hypotheses',
                           'graph paths do not establish literary interpretations']}
(RESULTS/'manifest.json').write_text(json.dumps(MANIFEST,indent=2,ensure_ascii=False),encoding='utf-8')
report=['# Literary Second Brain — Comparative Research Dossier','',
        f'**Source atlas:** {len(CATALOG)} physical notes / {len(MG)} graph nodes / {MG.number_of_edges()} directed multiedges',
        f'**LLM status:** {LLM_STATUS}','',
        '## Methodological caution','A graph path is an itinerary for reading, not proof that a book answers the question or influenced another work.','']
for qid,q,_ in QUESTIONS:
    report += [f'## {qid}: {q}','']
    for trip in [t for t in ALL_JOURNEYS if t['qid']==qid]:
        report += [f"**{trip['engine']}**: "+' → '.join(label(x) for x in trip['path']),
                   f"Works on route: {sum(simple_kind(x)=='obra' for x in trip['path'])}. See the linked Obsidian trace.",'']
    report += ['**Open question:** Which primary-text passages and credible scholarly sources would validate the proposed literary connections?','']
(RESULTS/'READABLE_RESEARCH_REPORT.md').write_text('\n'.join(report),encoding='utf-8')
display(Markdown('\n'.join(report[:45])))
FINAL_ZIP=ROOT/'TOPIC_795_ENHANCED_LITERARY_VAULT.zip'
with zipfile.ZipFile(FINAL_ZIP,'w',zipfile.ZIP_DEFLATED) as z:
    for p in VAULT.rglob('*'):
        if p.is_file():z.write(p,p.relative_to(WORK))
print('Report:',RESULTS/'READABLE_RESEARCH_REPORT.md')
print('Enhanced Obsidian vault ZIP:',FINAL_ZIP)
print('Audit manifest:',RESULTS/'manifest.json')

## Conclusion — Learning to distinguish a path through literature from a literary judgment

This laboratory began with a deliberately simple human problem: a reader has a growing collection of books and wants to use that collection to think more carefully about enduring questions. The original challenge was not to write code. It was to make the reader's intellectual map **visible and inspectable**, while resisting the temptation to treat resemblance, proximity, or algorithmic ranking as established interpretation.

We first learned that a vault and a knowledge graph are complementary, not identical. The Markdown records are the durable documents that people can edit; the stored multigraph is a machine-operable representation in which different directed relations may connect the same two notes. We examined both representations physically, preserved uncertainty statuses, and did not suppress the source atlas's warning about its incomplete list of works. Next, we used a cartographic metaphor to construct a manageable local map rather than staring at a dense visualization of every recorded connection.

The journey then divided into two methods. The deterministic explorer selected books according to a declared scoring policy. Its decisions are transparent, but the very transparency reveals the policy's limitations: the weights are chosen by the instructor, and a high-scoring route need not contain strong literary evidence. The Claude-guided explorer is more flexible in choosing among authorized neighboring notes, but it does not have authority to invent a record or to treat unsupported influence claims as genuine history. In both cases, the governance mechanism is as important as the apparent intelligence of the selection procedure.

Finally, our research packets preserved the wiring of the chosen paths: node IDs, direction of traversal, relation types, uncertainty statuses and excerpts from the vault notes. But the notebook refused to confuse those *graph-derived materials* with direct evidence from the published literary texts. This is not a retreat from AI capability. It is the foundation for responsible intellectual use. The right way to extend the experiment is to collect authenticated passages, scholarly criticism, and reader responses, then evaluate whether different navigation strategies actually improve completeness, insight, or reflection.

### Questions for discussion

1. If every book is tentatively connected to all 15 existential questions, does that improve discovery or contaminate it?
2. What evidence would convert an `hipotesis_pendiente` edge into a supported interpretive relationship?
3. Should a language model choose books based on thematic diversity, canonical importance, or an explicitly defined learning objective?
4. Does a two-hop route tell us anything about the strength of the underlying argument?
5. How could we evaluate *learning* rather than merely comparing graph metrics?

**The takeaway:** A successful literary Second Brain is not a machine that confidently answers life's questions because the books are connected. It is a disciplined research environment that helps its reader discover **which books to examine, why a connection may matter, what evidence supports it, and where uncertainty remains**.

### Selected reading on methods

- Vannevar Bush, “As We May Think,” *The Atlantic* (1945).
- J. D. Novak and A. J. Cañas, “The Theory Underlying Concept Maps and How to Construct and Use Them,” IHMC (2008).
- David A. Kolb, *Experiential Learning* (1984).
- David Jonassen, *Computers as Mindtools for Schools* (2000).
- R. S. Sutton and A. G. Barto, *Reinforcement Learning: An Introduction*, 2nd ed. (2018), for the distinction between policy and evaluation.
- J. Pearl and D. Mackenzie, *The Book of Why* (2018), for why relationships do not establish causality.

*This is a structural methodology tutorial. Its literary claims must ultimately be checked against primary texts and reliable bibliographic sources.*